# Model Setup

In [6]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json
import sys
import time

import torch
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from huggingface_hub import hf_hub_download

MOIRAI_root = "*/src"

os.chdir(MOIRAI_root); sys.path.insert(0, MOIRAI_root)


from uni2ts.eval_util.plot import plot_single
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

from uni2ts.model.moirai_moe import MoiraiMoEForecast, MoiraiMoEModule
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

os.chdir("*/1. Environment/")

from data_generator import make_walk
from dataset import build_dataset, load_overview, load_window
from metrics import conditional_mean_forecaster, naive_forecaster, squared_error, ratio_rmse_floor, skill_vs_naive, skill_vs_conditional, drift_mle, plugin_forecaster


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")
print([name for name, _ in module.named_children()])
print("num params:", sum(p.numel() for p in module.parameters()))

['scaler', 'in_proj', 'encoder', 'out_proj']
num params: 11387208


In [8]:
def forecast_moirai(context, horizon, model):
    context = np.asarray(context, dtype=float)

    past_target   = torch.tensor(context, dtype=torch.float32).reshape(1, -1, 1)
    past_observed = torch.ones_like(past_target, dtype=torch.bool)
    past_is_pad   = torch.zeros(1, len(context), dtype=torch.bool)

    with torch.no_grad():
        out = model(
            past_target=past_target,
            past_observed_target=past_observed,
            past_is_pad=past_is_pad,
        )

    quantiles  = out.numpy()[0]       # (9, horizon)
    prediction = quantiles[4]         # median

    if len(prediction) != horizon:
        raise ValueError(f"got {len(prediction)} values, expected {horizon}")

    return prediction

In [9]:
data_path = "*/Setup/"

In [10]:
overview_data = load_overview(data_path)


ORIGIN      = int(overview_data["origin"].iloc[0])
MAX_HORIZON = int(overview_data["max_horizon"].iloc[0])
HORIZONS    = range(1, MAX_HORIZON + 1)
SIGMA       = float(overview_data["sigma"].iloc[0])

MOIRAI_VARIANT = "moirai2"     # or "moirai", "moirai-moe"
MOIRAI_SIZE    = "large"
MOIRAI_PATCH   = 32            # fixed across all conditions — see below
NUM_SAMPLES    = 100

In [18]:
MOIRAI_VARIANT = "moirai-2.0-R-small"
NUM_SAMPLES    = 1000


def make_moirai(context_length, horizon):
    model = Moirai2Forecast(
        module=Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small"),
        prediction_length=horizon,
        context_length=context_length,
        target_dim=1,
        feat_dynamic_real_dim=0,
        past_feat_dynamic_real_dim=0,
    )
    return model


MOIRAI_MODELS = {L: make_moirai(L, MAX_HORIZON) for L in [32, 64, 128, 256,512]}
moirai_model = MOIRAI_MODELS[256]


MOIRAI_MODELS = {
    L: make_moirai(L, MAX_HORIZON) for L in [32, 64, 128, 256,512]
}


In [19]:
overview_data

,walk_id,snr,seed,sigma,mu,origin,max_horizon
0,snr0.01_seed0,0.01,0,1.0,0.01,512,256
1,snr0.01_seed1,0.01,1,1.0,0.01,512,256
2,snr0.01_seed2,0.01,2,1.0,0.01,512,256
3,snr0.01_seed3,0.01,3,1.0,0.01,512,256
4,snr0.01_seed4,0.01,4,1.0,0.01,512,256
...,...,...,...,...,...,...,...
12495,snr5_seed95,5.00,95,1.0,5.00,512,256
12496,snr5_seed96,5.00,96,1.0,5.00,512,256
12497,snr5_seed97,5.00,97,1.0,5.00,512,256
12498,snr5_seed98,5.00,98,1.0,5.00,512,256


# Implementation

## L = 512

In [20]:
CONTEXT_LEN = 512

In [21]:
rows_context_512 = []
moirai_model = MOIRAI_MODELS[CONTEXT_LEN]

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)
    prediction  = forecast_moirai(context, MAX_HORIZON, moirai_model)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_512.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "moirai",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_512 = pd.DataFrame(rows_context_512)

12000/12500

In [22]:
predictions_context_512

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,512,1,moirai,0.152914,-0.010026,0.0,-0.014330,0.01,0.026549,2.338258e-02,0.027970,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,512,2,moirai,-0.381044,-0.046351,0.0,-0.028660,0.02,0.112019,1.451946e-01,0.124175,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,512,3,moirai,-0.504496,-0.040942,0.0,-0.042990,0.03,0.214882,2.545158e-01,0.212988,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,512,4,moirai,0.803376,-0.037091,0.0,-0.057320,0.04,0.706385,6.454132e-01,0.740797,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,512,5,moirai,-0.154444,0.007133,0.0,-0.071649,0.05,0.026107,2.385287e-02,0.006855,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,512,252,moirai,1260.902970,1598.276733,0.0,1265.948489,1260.00,113821.056566,1.589876e+06,25.457272,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,512,253,moirai,1263.939192,1602.019409,0.0,1270.972095,1265.00,114298.233189,1.597542e+06,49.461718,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,512,254,moirai,1268.951206,1609.724976,0.0,1275.995700,1270.00,116126.762011,1.610237e+06,49.624891,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,512,255,moirai,1273.053372,1618.129028,0.0,1281.019305,1275.00,119077.208339,1.620665e+06,63.456080,3.789359


In [23]:

predictions_context_512.to_csv("*/Moirai/predicitons_512_moirai.csv")


## L = 256

In [18]:
CONTEXT_LEN = 256

In [19]:
rows_context_256 = []
moirai_model = MOIRAI_MODELS[CONTEXT_LEN]

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)
    prediction  = forecast_moirai(context, MAX_HORIZON, moirai_model)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_256.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "moirai",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_256 = pd.DataFrame(rows_context_256)

12000/12500

In [20]:
predictions_context_256

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,256,1,moirai,0.152914,0.036641,0.0,-0.040656,0.01,0.013519,2.338258e-02,0.037469,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,256,2,moirai,-0.381044,0.049561,0.0,-0.081313,0.02,0.185420,1.451946e-01,0.089839,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,256,3,moirai,-0.504496,0.088186,0.0,-0.121969,0.03,0.351271,2.545158e-01,0.146327,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,256,4,moirai,0.803376,0.076338,0.0,-0.162625,0.04,0.528584,6.454132e-01,0.933158,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,256,5,moirai,-0.154444,0.133563,0.0,-0.203281,0.05,0.082948,2.385287e-02,0.002385,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,256,252,moirai,1260.902970,1495.581299,0.0,1265.174318,1260.00,55073.918255,1.589876e+06,18.244415,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,256,253,moirai,1263.939192,1501.847656,0.0,1270.194851,1265.00,56600.437320,1.597542e+06,39.133264,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,256,254,moirai,1268.951206,1510.282593,0.0,1275.215384,1270.00,58240.838223,1.610237e+06,39.239921,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,256,255,moirai,1273.053372,1517.329346,0.0,1280.235917,1275.00,59670.751154,1.620665e+06,51.588943,3.789359


In [21]:

predictions_context_256.to_csv("*/Moirai/predicitons_256_moirai.csv")


## L = 128

In [22]:
CONTEXT_LEN = 128

In [23]:
rows_context_128 = []
moirai_model = MOIRAI_MODELS[CONTEXT_LEN]

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)
    prediction  = forecast_moirai(context, MAX_HORIZON, moirai_model)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_128.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "moirai",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_128 = pd.DataFrame(rows_context_128)

12000/12500

In [24]:
predictions_context_128

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,128,1,moirai,0.152914,0.026620,0.0,0.011189,0.01,0.015950,2.338258e-02,0.020086,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,128,2,moirai,-0.381044,-0.068332,0.0,0.022378,0.02,0.097789,1.451946e-01,0.162749,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,128,3,moirai,-0.504496,-0.030023,0.0,0.033567,0.03,0.225124,2.545158e-01,0.289511,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,128,4,moirai,0.803376,0.012179,0.0,0.044756,0.04,0.625992,6.454132e-01,0.575505,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,128,5,moirai,-0.154444,0.113220,0.0,0.055945,0.05,0.071644,2.385287e-02,0.044263,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,128,252,moirai,1260.902970,1238.247437,0.0,1291.282869,1260.00,513.273175,1.589876e+06,922.938322,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,128,253,moirai,1263.939192,1244.215454,0.0,1296.407008,1265.00,389.025840,1.597542e+06,1054.159060,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,128,254,moirai,1268.951206,1249.533081,0.0,1301.531146,1270.00,377.063578,1.610237e+06,1061.452502,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,128,255,moirai,1273.053372,1254.234863,0.0,1306.655285,1275.00,354.136284,1.620665e+06,1129.088505,3.789359


In [25]:

predictions_context_128.to_csv("*/Moirai/predicitons_128_moirai.csv")


## L = 64

In [26]:
CONTEXT_LEN = 64

In [27]:
rows_context_64 = []
moirai_model = MOIRAI_MODELS[CONTEXT_LEN]

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)
    prediction  = forecast_moirai(context, MAX_HORIZON, moirai_model)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_64.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "moirai",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_64 = pd.DataFrame(rows_context_64)

12000/12500

In [28]:
predictions_context_64

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,64,1,moirai,0.152914,0.167924,0.0,-0.102400,0.01,0.000225,2.338258e-02,0.065185,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,64,2,moirai,-0.381044,0.211441,0.0,-0.204800,0.02,0.351039,1.451946e-01,0.031062,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,64,3,moirai,-0.504496,0.254256,0.0,-0.307200,0.03,0.575704,2.545158e-01,0.038926,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,64,4,moirai,0.803376,0.256199,0.0,-0.409600,0.04,0.299403,6.454132e-01,1.471311,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,64,5,moirai,-0.154444,0.287624,0.0,-0.512000,0.05,0.195424,2.385287e-02,0.127846,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,64,252,moirai,1260.902970,911.643188,0.0,1310.411462,1260.00,121982.394642,1.589876e+06,2451.090874,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,64,253,moirai,1263.939192,914.521362,0.0,1315.611508,1265.00,122092.819771,1.597542e+06,2670.028225,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,64,254,moirai,1268.951206,916.015869,0.0,1320.811553,1270.00,124563.352031,1.610237e+06,2689.495630,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,64,255,moirai,1273.053372,917.982544,0.0,1326.011599,1275.00,126075.293189,1.620665e+06,2804.573759,3.789359


In [29]:

predictions_context_64.to_csv("*/Moirai/predicitons_64_moirai.csv")


## L = 32

In [30]:
CONTEXT_LEN = 32

In [31]:
rows_context_32 = []
moirai_model = MOIRAI_MODELS[CONTEXT_LEN]

for i in range(len(overview_data)):
    walk = overview_data.iloc[i]

    context, future = load_window(
        walk["walk_id"], CONTEXT_LEN, ORIGIN, MAX_HORIZON, data_path
    )

    if i % 500 == 0:
        print(f"\r{i}/{len(overview_data)}", end="", flush=True)
    prediction  = forecast_moirai(context, MAX_HORIZON, moirai_model)
    plugin      = plugin_forecaster(context, MAX_HORIZON)
    conditional = conditional_mean_forecaster(context, MAX_HORIZON, walk["mu"])
    naive       = naive_forecaster(context, MAX_HORIZON)

    for h in HORIZONS:
        rows_context_32.append({
            "walk_id":     walk["walk_id"],
            "snr":         walk["snr"],
            "seed":        walk["seed"],
            "sigma":       walk["sigma"],
            "mu":          walk["mu"],
            "context_len": CONTEXT_LEN,
            "horizon":     h,
            "model":       "moirai",
            "actual":      future[h - 1],
            "forecast":    prediction[h - 1],
            "naive":       naive[h - 1],
            "plugin":      plugin[h - 1],
            "conditional": conditional[h - 1],
            "se_model":       squared_error(prediction,  future, h),
            "se_naive":       squared_error(naive,       future, h),
            "se_plugin":      squared_error(plugin,      future, h),
            "se_conditional": squared_error(conditional, future, h),
        })

predictions_context_32 = pd.DataFrame(rows_context_32)

12000/12500

In [32]:
predictions_context_32

,walk_id,snr,seed,sigma,mu,context_len,horizon,model,actual,forecast,naive,plugin,conditional,se_model,se_naive,se_plugin,se_conditional
0,snr0.01_seed0,0.01,0,1.0,0.01,32,1,moirai,0.152914,0.228233,0.0,-0.023217,0.01,5.673024e-03,2.338258e-02,0.031022,0.020424
1,snr0.01_seed0,0.01,0,1.0,0.01,32,2,moirai,-0.381044,0.276885,0.0,-0.046434,0.02,4.328712e-01,1.451946e-01,0.111964,0.160836
2,snr0.01_seed0,0.01,0,1.0,0.01,32,3,moirai,-0.504496,0.340406,0.0,-0.069650,0.03,7.138592e-01,2.545158e-01,0.189090,0.285686
3,snr0.01_seed0,0.01,0,1.0,0.01,32,4,moirai,0.803376,0.380086,0.0,-0.092867,0.04,1.791748e-01,6.454132e-01,0.803252,0.582743
4,snr0.01_seed0,0.01,0,1.0,0.01,32,5,moirai,-0.154444,0.389350,0.0,-0.116084,0.05,2.957115e-01,2.385287e-02,0.001471,0.041797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3199995,snr5_seed99,5.00,99,1.0,5.00,32,252,moirai,1260.902970,39.374390,0.0,1309.016155,1260.00,1.492132e+06,1.589876e+06,2314.878649,0.815354
3199996,snr5_seed99,5.00,99,1.0,5.00,32,253,moirai,1263.939192,39.829712,0.0,1314.210664,1265.00,1.498444e+06,1.597542e+06,2527.220875,1.125313
3199997,snr5_seed99,5.00,99,1.0,5.00,32,254,moirai,1268.951206,38.826042,0.0,1319.405172,1270.00,1.513208e+06,1.610237e+06,2545.602723,1.099969
3199998,snr5_seed99,5.00,99,1.0,5.00,32,255,moirai,1273.053372,35.523315,0.0,1324.599681,1275.00,1.531481e+06,1.620665e+06,2657.021932,3.789359


In [33]:

predictions_context_32.to_csv("*/Moirai/predicitons_32_moirai.csv")
